In [1]:
%pylab inline

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


In [2]:
start_time = 1187008769
end_time = 1187008886
seg_len = 16
sample_rate = 512
seg_start_pad = 4
seg_end_pad = 2
trigger_start = int(start_time) + seg_start_pad
trigger_end = int(end_time) - seg_end_pad
data_start = (trigger_start - seg_start_pad) - int(start_time)
data_end = trigger_end + seg_end_pad - int(start_time)
data_dur = data_end - data_start
data_start *= sample_rate
data_end *= sample_rate
analyzable = trigger_end - trigger_start
throwaway_size = seg_start_pad + seg_end_pad
seg_width = seg_len - throwaway_size

In [5]:
def calculate_segments(segment_overlap):
    #number of segments we need to analyze this data
    num_segs = int(numpy.ceil(float(analyzable) / float(seg_width + segment_overlap)))
    # The offset we will use between segments
    seg_offset = int(numpy.ceil(analyzable / float(num_segs)))
    segment_slices = []
    analyze_slices = []

    # Determine how to chop up the strain into smaller segments
    for nseg in range(num_segs-1):
        # boundaries for time slices into the strain
        seg_start = int(data_start + nseg * (seg_offset - segment_overlap) * sample_rate)
        seg_end = int(seg_start + seg_len * sample_rate)
        seg_slice = slice(seg_start, seg_end)
        segment_slices.append(seg_slice)

        # boundaries for the analyzable portion of the segment
        ana_start = int(seg_start_pad * sample_rate)
        ana_end = int(ana_start + seg_offset * sample_rate)
        ana_slice = slice(ana_start, ana_end)
        analyze_slices.append(ana_slice)

    # The last segment takes up any integer boundary slop
    seg_end = int(data_end)
    seg_start = int(seg_end - seg_len * sample_rate)
    seg_slice = slice(seg_start, seg_end)
    segment_slices.append(seg_slice)

    remaining = (data_dur - ((num_segs - 1) * (seg_offset - segment_overlap) + seg_start_pad))
    ana_start = int((seg_len - remaining) * sample_rate)
    ana_end = int((seg_len - seg_end_pad) * sample_rate)
    ana_slice = slice(ana_start, ana_end)
    analyze_slices.append(ana_slice)
    return analyze_slices, segment_slices

In [8]:
calculate_segments(segment_overlap=17/sample_rate)

([slice(2048, 7168, None),
  slice(2048, 7168, None),
  slice(2048, 7168, None),
  slice(2048, 7168, None),
  slice(2048, 7168, None),
  slice(2048, 7168, None),
  slice(2048, 7168, None),
  slice(2048, 7168, None),
  slice(2048, 7168, None),
  slice(2048, 7168, None),
  slice(2048, 7168, None),
  slice(6469, 7168, None)],
 [slice(0, 8192, None),
  slice(5103, 13295, None),
  slice(10206, 18398, None),
  slice(15309, 23501, None),
  slice(20412, 28604, None),
  slice(25515, 33707, None),
  slice(30618, 38810, None),
  slice(35721, 43913, None),
  slice(40824, 49016, None),
  slice(45927, 54119, None),
  slice(51030, 59222, None),
  slice(51712, 59904, None)])

In [9]:
calculate_segments(segment_overlap=0)

([slice(2048, 7168, None),
  slice(2048, 7168, None),
  slice(2048, 7168, None),
  slice(2048, 7168, None),
  slice(2048, 7168, None),
  slice(2048, 7168, None),
  slice(2048, 7168, None),
  slice(2048, 7168, None),
  slice(2048, 7168, None),
  slice(2048, 7168, None),
  slice(2048, 7168, None),
  slice(6656, 7168, None)],
 [slice(0, 8192, None),
  slice(5120, 13312, None),
  slice(10240, 18432, None),
  slice(15360, 23552, None),
  slice(20480, 28672, None),
  slice(25600, 33792, None),
  slice(30720, 38912, None),
  slice(35840, 44032, None),
  slice(40960, 49152, None),
  slice(46080, 54272, None),
  slice(51200, 59392, None),
  slice(51712, 59904, None)])

In [10]:
51200 - 51030

170

In [11]:
arange(51030, 59222)[7168], arange(51200, 59392)[7168]

(58198, 58368)

In [12]:
arange(51712, 59904)[slice(6469, 7168, None)][0], arange(51712, 59904)[slice(6656, 7168, None)][0]

(58181, 58368)

In [13]:
(num_segs-2) * 17

170